<a href="https://colab.research.google.com/github/dhillonarsh-ai/capstone-project-NoShowGuard-/blob/main/notebooks/Part1_Data_Cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [101]:
import urllib.request

dataset_url = (
    "https://raw.githubusercontent.com/"
    "dhillonarsh-ai/capstone-project-NoShowGuard-/"
    "main/data/raw/appointments.csv"
)

urllib.request.urlretrieve(dataset_url, "appointments.csv")

print("Dataset downloaded successfully.")


Dataset downloaded successfully.


In [102]:
import csv

file_path = "appointments.csv"

with open(file_path, "r", newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)

    print("Column names:")
    print(reader.fieldnames)

    rows = list(reader)

    print("\nTotal records:", len(rows))

Column names:
['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']

Total records: 1020


## 1.1 - CSV Data Handling & Initial Validation
In this section, the raw appointment dataset is processed using Python's built-in `csv` module. The objective is to understand the structure of the dataset, read appointment records, validate individual data fields, and identify records that cannot be processed correctly.


In [103]:
#Part 1.1: Data Validation

def parse_row(row):
    """Convert required numeric fields into appropriate data types."""

    row["age"] = int(row["age"])
    row["fee_inr"] = int(row["fee_inr"])
    row["prior_no_shows"] = int(row["prior_no_shows"])
    row["distance_km"] = float(row["distance_km"])

    return row


In [104]:
# Validate CSV records
accepted_rows = []
rejected_rows = []

with open("appointments.csv", "r", newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)

    for row in reader:
        try:
            parsed_row = parse_row(row)
            accepted_rows.append(parsed_row)

        except ValueError:
            rejected_rows.append(row)

print("Accepted records:", len(accepted_rows))
print("Rejected records:", len(rejected_rows))
print("Total processed:", len(accepted_rows) + len(rejected_rows))


Accepted records: 988
Rejected records: 32
Total processed: 1020


In [105]:
# Save rejected records as JSON

import json

with open("rejected_rows.json", "w", encoding="utf-8") as file:
    json.dump(rejected_rows, file, indent=2)

print("Rejected records saved successfully.")
print("Rejected records:", len(rejected_rows))


Rejected records saved successfully.
Rejected records: 32



## Part 1.2 - Data Exploration & Initial Cleaning Using Pandas
In this section, the appointment dataset is explored and cleaned using the Pandas library. The focus is on understanding the dataset's structure, identifying missing values and duplicate records, standardizing categorical values, and handling invalid date entries.


In [106]:

import pandas as pd
import numpy as np

df = pd.read_csv("appointments.csv")

print("Dataset shape:", df.shape)
print("\nDataset information:")
df.info()


Dataset shape: (1020, 14)

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   object 
 1   patient_id       1020 non-null   object 
 2   clinic_id        1020 non-null   object 
 3   booked_on        1020 non-null   object 
 4   appt_date        1019 non-null   object 
 5   age              1008 non-null   float64
 6   gender           1020 non-null   object 
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   object 
 10  visit_type       1020 non-null   object 
 11  reminder_sent    1020 non-null   object 
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   object 
dtypes: float64(2), int64(2), object(10)
memory usage: 111.7+ KB


In [107]:
# Data Quality Inspection

print("Missing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

Missing values:
appointment_id      0
patient_id          0
clinic_id           0
booked_on           0
appt_date           1
age                12
gender              0
distance_km        20
fee_inr             0
booking_channel     0
visit_type          0
reminder_sent       0
prior_no_shows      0
status              4
dtype: int64

Duplicate rows: 20


In [108]:
# Removing duplicates and standardize booking channels

# Step 1: Remove exact duplicate rows
df = df.drop_duplicates()

print("After removing duplicates:", df.shape)

# Step 2: Standardize booking_channel
df["booking_channel"] = (
    df["booking_channel"]
    .str.strip()
    .str.title()
)

print("\nBooking channel categories:")
print(df["booking_channel"].value_counts())


After removing duplicates: (1000, 14)

Booking channel categories:
booking_channel
App        387
Phone      359
Walk-In    254
Name: count, dtype: int64


In [109]:
# Convert dates and remove invalid records

# Convert date columns
df["appt_date"] = pd.to_datetime(
    df["appt_date"], errors="coerce"
)

df["booked_on"] = pd.to_datetime(
    df["booked_on"], errors="coerce"
)

# Remove rows with missing appointment date or status
df = df.dropna(subset=["appt_date", "status"])

print("Final cleaned shape:", df.shape)
print("Missing appointment dates:", df["appt_date"].isnull().sum())
print("Missing statuses:", df["status"].isnull().sum())

Final cleaned shape: (990, 14)
Missing appointment dates: 0
Missing statuses: 0


## 1.3 - Missing Value Imputation & Outlier Treatment

In this section, we improve the quality and consistency of the appointment dataset by handling invalid values, imputing missing data, and treating statistical outliers.

In [110]:

# Identify invalid age values
invalid_age_mask = (df["age"] < 1) | (df["age"] > 100)

# Count invalid age records
invalid_age_count = invalid_age_mask.sum()

# Replace invalid ages with NaN
df.loc[invalid_age_mask, "age"] = np.nan

print("Invalid age values identified:", invalid_age_count)
print("Missing age values after replacement:", df["age"].isna().sum())


Invalid age values identified: 5
Missing age values after replacement: 17


In [111]:
# Impute missing age values using median

# Calculate median age
age_median = df["age"].median()

# Replace missing ages with median age
df["age"] = df["age"].fillna(age_median)

# Round and convert age into integer
df["age"] = df["age"].round().astype(int)

print("Median age:", age_median)
print("Missing ages after imputation:", df["age"].isna().sum())
print("Age data type:", df["age"].dtype)

Median age: 48.0
Missing ages after imputation: 0
Age data type: int64


In [112]:
# Impute missing distance values

# Calculate median distance
distance_median = df["distance_km"].median()

# Fill missing distance values using median
df["distance_km"] = df["distance_km"].fillna(distance_median)

print("Median distance:", distance_median)
print("Missing distances after imputation:", df["distance_km"].isna().sum())

Median distance: 8.25
Missing distances after imputation: 0


In [113]:
# Calculating IQR and identify outliers in distance_km

# Calculate quartiles and IQR for distance
distance_q1 = df["distance_km"].quantile(0.25)
distance_q3 = df["distance_km"].quantile(0.75)

distance_iqr = distance_q3 - distance_q1

# Calculate outlier boundaries
distance_lower = distance_q1 - (1.5 * distance_iqr)
distance_upper = distance_q3 + (1.5 * distance_iqr)

# Identify outliers
distance_outlier_mask = (
    (df["distance_km"] < distance_lower) |
    (df["distance_km"] > distance_upper)
)

distance_outlier_count = distance_outlier_mask.sum()

print("Distance Q1:", distance_q1)
print("Distance Q3:", distance_q3)
print("Distance IQR:", distance_iqr)
print("Lower Bound:", distance_lower)
print("Upper Bound:", distance_upper)
print("Distance Outliers:", distance_outlier_count)


Distance Q1: 4.625
Distance Q3: 13.0
Distance IQR: 8.375
Lower Bound: -7.9375
Upper Bound: 25.5625
Distance Outliers: 26


In [114]:
# Detect outliers in fee_inr

# Calculate quartiles and IQR for consultation fee
fee_q1 = df["fee_inr"].quantile(0.25)
fee_q3 = df["fee_inr"].quantile(0.75)

fee_iqr = fee_q3 - fee_q1

# Calculate outlier boundaries
fee_lower = fee_q1 - (1.5 * fee_iqr)
fee_upper = fee_q3 + (1.5 * fee_iqr)

# Identify outliers
fee_outlier_mask = (
    (df["fee_inr"] < fee_lower) |
    (df["fee_inr"] > fee_upper)
)

fee_outlier_count = fee_outlier_mask.sum()

print("Fee Q1:", fee_q1)
print("Fee Q3:", fee_q3)
print("Fee IQR:", fee_iqr)
print("Lower Bound:", fee_lower)
print("Upper Bound:", fee_upper)
print("Fee Outliers:", fee_outlier_count)


Fee Q1: 500.0
Fee Q3: 787.5
Fee IQR: 287.5
Lower Bound: 68.75
Upper Bound: 1218.75
Fee Outliers: 3


In [115]:
# Cap the outliers using np.clip()

# Cap distance values within IQR boundaries
df["distance_km"] = np.clip(
    df["distance_km"],
    distance_lower,
    distance_upper
)

# Cap consultation fees within IQR boundaries
df["fee_inr"] = np.clip(
    df["fee_inr"],
    fee_lower,
    fee_upper
)

# Round distance to one decimal place
df["distance_km"] = df["distance_km"].round(1)

# Round fees and convert to integer
df["fee_inr"] = df["fee_inr"].round().astype(int)

print("Distance values capped:", distance_outlier_count)
print("Fee values capped:", fee_outlier_count)
print("Maximum distance after capping:", df["distance_km"].max())
print("Maximum fee after capping:", df["fee_inr"].max())


Distance values capped: 26
Fee values capped: 3
Maximum distance after capping: 25.6
Maximum fee after capping: 1219


In [116]:

# Validate cleaned numerical columns
print("Missing age values:", df["age"].isna().sum())
print("Missing distance values:", df["distance_km"].isna().sum())
print("Missing fee values:", df["fee_inr"].isna().sum())

print("\nFinal Data Types:")
print(df[["age", "distance_km", "fee_inr"]].dtypes)

print("\nFinal Dataset Shape:", df.shape)


Missing age values: 0
Missing distance values: 0
Missing fee values: 0

Final Data Types:
age              int64
distance_km    float64
fee_inr          int64
dtype: object

Final Dataset Shape: (990, 14)


## 1.4 - Feature Engineering & Statistical Analysis

In this section, additional analytical features are derived from the cleaned appointment dataset. The focus is on calculating appointment lead time, converting appointment outcomes into a numerical format, and performing statistical analysis using NumPy.

In [117]:

# Calculate the number of days between booking and appointment
df["lead_days"] = (df["appt_date"] - df["booked_on"]).dt.days

# Display lead time statistics
print("Minimum lead days:", df["lead_days"].min())
print("Maximum lead days:", df["lead_days"].max())
print("Average lead days:", df["lead_days"].mean())


Minimum lead days: 1
Maximum lead days: 45
Average lead days: 11.819191919191919


In [118]:

# Convert appointment status into a binary target variable
df["no_show"] = (df["status"] == "No-Show").astype(int)

# Check no-show distribution
print(df["no_show"].value_counts())

# Calculate total no-shows and percentage
total_no_shows = df["no_show"].sum()
no_show_percentage = df["no_show"].mean() * 100

print("Total no-shows:", total_no_shows)
print("No-show percentage:", round(no_show_percentage, 1), "%")


no_show
0    772
1    218
Name: count, dtype: int64
Total no-shows: 218
No-show percentage: 22.0 %


In [119]:

# Extract numerical columns into a NumPy array
X = df[["age", "distance_km", "fee_inr", "lead_days"]].to_numpy()

# Calculate mean across columns
feature_means = np.mean(X, axis=0)

print("Feature means:", feature_means)


Feature means: [ 48.17373737   9.50090909 635.71414141  11.81919192]


In [120]:

# Identify appointments with lead time greater than 14 days
long_lead_mask = df["lead_days"] > 14

# Calculate count and proportion
long_lead_count = long_lead_mask.sum()
long_lead_share = long_lead_mask.mean() * 100

print("Appointments with lead days > 14:", long_lead_count)
print("Percentage of long-lead appointments:", round(long_lead_share, 1), "%")


Appointments with lead days > 14: 296
Percentage of long-lead appointments: 29.9 %


In [121]:

# Calculate distance Z-scores using population standard deviation
distance_z = (
    df["distance_km"] - np.mean(df["distance_km"])
) / np.std(df["distance_km"])

# Identify observations with absolute Z-score greater than 2
distance_z_outlier_mask = np.abs(distance_z) > 2

# Count statistical outliers
distance_z_outlier_count = distance_z_outlier_mask.sum()

print("Distance Z-score outliers:", distance_z_outlier_count)


Distance Z-score outliers: 53


In [122]:

# Validate newly created features
print("New columns:", ["lead_days", "no_show"])
print("Dataset shape:", df.shape)
print("Missing lead days:", df["lead_days"].isna().sum())
print("Missing no_show values:", df["no_show"].isna().sum())

# Preview newly created features
df[["appointment_id", "lead_days", "status", "no_show"]].head()


New columns: ['lead_days', 'no_show']
Dataset shape: (990, 16)
Missing lead days: 0
Missing no_show values: 0


,appointment_id,lead_days,status,no_show
0,A0001,1,No-Show,1
1,A0002,10,Attended,0
2,A0003,6,Attended,0
4,A0004,4,Attended,0
5,A0005,40,No-Show,1


##1.5 - External API Integration & Clinic Data Enrichment


In this section, external weather data is integrated into the clinic dataset using the Open-Meteo API. The objective is to retrieve temperature information for each clinic based on its geographical coordinates and enrich the existing clinic records with additional environmental information.

In [123]:
import urllib.request
import requests
import pandas as pd
import time

# Download the clinics.csv dataset
clinics_url = (
    "https://raw.githubusercontent.com/"
    "dhillonarsh-ai/capstone-project-NoShowGuard-/"
    "main/data/raw/clinics.csv"
)
urllib.request.urlretrieve(clinics_url, "clinics.csv")
print("Clinics dataset downloaded successfully.")

# Load the original clinic dataset
clinics = pd.read_csv("clinics.csv")

# Inspect clinic data
print("Clinic dataset shape:", clinics.shape)
display(clinics.head())

Clinics dataset downloaded successfully.
Clinic dataset shape: (8, 6)


,clinic_id,clinic_name,city,tier,latitude,longitude
0,C01,Kothrud Family Clinic,Pune,Metro,18.5074,73.8077
1,C02,Hadapsar Care Centre,Pune,Metro,18.5089,73.9260
2,C03,Sitabuldi Health Point,Nagpur,Tier-2,21.1458,79.0882
3,C04,Malviya Nagar Clinic,Jaipur,Tier-2,26.8553,75.8150
4,C05,Vaishali Nagar Clinic,Jaipur,Tier-2,26.9124,75.7873


In [124]:

# Select the first clinic
clinic = clinics.iloc[0]

# Open-Meteo API endpoint
url = "https://api.open-meteo.com/v1/forecast"

# API parameters
params = {
    "latitude": clinic["latitude"],
    "longitude": clinic["longitude"],
    "hourly": "temperature_2m",
    "forecast_days": 1
}

# Send GET request
response = requests.get(url, params=params, timeout=10)

print("Status code:", response.status_code)

if response.status_code == 200:
    weather_data = response.json()
    print("Temperature:", weather_data["hourly"]["temperature_2m"][0])


Status code: 200
Temperature: 20.9


In [125]:

# Function to fetch temperature for a clinic
def fetch_temperature(latitude, longitude):
    url = "https://api.open-meteo.com/v1/forecast"

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "hourly": "temperature_2m",
        "forecast_days": 1
    }

    for attempt in range(2):
        try:
            response = requests.get(url, params=params, timeout=10)

            # Raise an error for unsuccessful HTTP responses
            response.raise_for_status()

            weather_data = response.json()

            # Extract first hourly temperature
            temperatures = weather_data.get("hourly", {}).get("temperature_2m")

            if temperatures:
                return temperatures[0]

            print("Temperature unavailable for coordinates:", latitude, longitude)
            return None

        except requests.RequestException as error:
            if attempt == 0:
                print("Retrying request:", error)
                time.sleep(1)
            else:
                print("API request failed:", error)
                return None


In [126]:

# Fetch temperature for every clinic
clinics["temp_c"] = clinics.apply(
    lambda row: fetch_temperature(
        row["latitude"],
        row["longitude"]
    ),
    axis=1
)

# Inspect enriched clinic data
display(clinics.head())

print("Total clinics:", len(clinics))
print("Temperature values available:", clinics["temp_c"].notna().sum())


,clinic_id,clinic_name,city,tier,latitude,longitude,temp_c
0,C01,Kothrud Family Clinic,Pune,Metro,18.5074,73.8077,20.9
1,C02,Hadapsar Care Centre,Pune,Metro,18.5089,73.9260,21.0
2,C03,Sitabuldi Health Point,Nagpur,Tier-2,21.1458,79.0882,21.8
3,C04,Malviya Nagar Clinic,Jaipur,Tier-2,26.8553,75.8150,19.8
4,C05,Vaishali Nagar Clinic,Jaipur,Tier-2,26.9124,75.7873,19.8


Total clinics: 8
Temperature values available: 8


In [127]:

# Save enriched clinic dataset
clinics.to_csv("clinics_enriched.csv", index=False)

print("Clinic dataset exported successfully.")


Clinic dataset exported successfully.


##1.6 - Final Dataset Preparation & SQL Export

Overview

In this section, the cleaned appointment dataset is finalized and prepared for database integration. The focus is on maintaining a consistent column structure, validating record completeness, exporting the processed dataset, and generating SQL statements for loading data into MySQL.

In [128]:

# Sort appointments by appointment ID
df = df.sort_values("appointment_id").reset_index(drop=True)

# Format dates as YYYY-MM-DD
df["booked_on"] = df["booked_on"].dt.strftime("%Y-%m-%d")
df["appt_date"] = df["appt_date"].dt.strftime("%Y-%m-%d")

print("Appointments sorted and dates formatted.")


Appointments sorted and dates formatted.


In [129]:

# Define required column order
final_columns = [
    "appointment_id",
    "patient_id",
    "clinic_id",
    "booked_on",
    "appt_date",
    "age",
    "gender",
    "distance_km",
    "fee_inr",
    "booking_channel",
    "visit_type",
    "reminder_sent",
    "prior_no_shows",
    "status",
    "lead_days",
    "no_show"
]

# Arrange columns
df = df[final_columns]

print("Final columns:", df.columns.tolist())
print("Final dataset shape:", df.shape)


Final columns: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status', 'lead_days', 'no_show']
Final dataset shape: (990, 16)


In [130]:

# Check unique appointment IDs
unique_appointments = df["appointment_id"].is_unique

# Check missing values
missing_values = df.isna().sum().sum()

print("Unique appointment IDs:", unique_appointments)
print("Total missing values:", missing_values)
print("Total appointment records:", len(df))


Unique appointment IDs: True
Total missing values: 0
Total appointment records: 990


In [131]:

# Export final cleaned appointment dataset
df.to_csv("appointments_clean.csv", index=False)

print("appointments_clean.csv exported successfully.")


appointments_clean.csv exported successfully.


In [132]:

# SQL file generation
with open("load_data.sql", "w", encoding="utf-8") as sql_file:

    # Insert clinic records first
    for _, row in clinics.iterrows():

        temp_value = (
            "NULL" if pd.isna(row["temp_c"])
            else str(row["temp_c"])
        )

        sql_file.write(
            f"INSERT INTO clinics "
            f"(clinic_id, clinic_name, city, latitude, longitude, temp_c) "
            f"VALUES ('{row['clinic_id']}', "
            f"'{row['clinic_name']}', "
            f"'{row['city']}', "
            f"{row['latitude']}, "
            f"{row['longitude']}, "
            f"{temp_value});\n"
        )

    # Insert appointment records
    for _, row in df.iterrows():

        sql_file.write(
            f"INSERT INTO appointments "
            f"({', '.join(final_columns)}) "
            f"VALUES ("
            f"'{row['appointment_id']}', "
            f"'{row['patient_id']}', "
            f"'{row['clinic_id']}', "
            f"'{row['booked_on']}', "
            f"'{row['appt_date']}', "
            f"{row['age']}, "
            f"'{row['gender']}', "
            f"{row['distance_km']}, "
            f"{row['fee_inr']}, "
            f"'{row['booking_channel']}', "
            f"'{row['visit_type']}', "
            f"'{row['reminder_sent']}', "
            f"{row['prior_no_shows']}, "
            f"'{row['status']}', "
            f"{row['lead_days']}, "
            f"{row['no_show']});\n"
        )

print("load_data.sql generated successfully.")


load_data.sql generated successfully.


In [133]:

import os

# Check exported files
for filename in ["appointments_clean.csv", "clinics_enriched.csv", "load_data.sql"]:
    print(filename, "->", os.path.exists(filename))


appointments_clean.csv -> True
clinics_enriched.csv -> True
load_data.sql -> True
